In [52]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [53]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [54]:
import requests
from minsearch import Index

In [55]:
def build_index(documents):
    index = Index(
    text_fields=["content"])
    index.fit(documents)
    return index

In [56]:
index = build_index(documents)

In [57]:
def search(query):

    return index.search(
        query,
        num_results=10
    )

In [64]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the lessons."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [59]:
import json
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [71]:
from dotenv import load_dotenv
load_dotenv()
from openai import OpenAI
import os
openai_client = OpenAI(
    api_key="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",
    base_url="https://openrouter.ai/api/v1",
)

In [76]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model="openrouter/free",
            input=messages,
           tools=[search_tool]
        )
        #response = openai_client.chat.completions.create(
        #model="llama-3.3-70b-versatile",
        #input = messages,
        #tools = [search_tool],
        #)
        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [77]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using lessons, don't do it yourself. Only use the 
facts from the lessons database.

At the end, ask if there are other areas that the user wants to explore.
"""

In [78]:
question = "what's queen gambit?"

result = agent_loop(instructions, question)

iteration #1...


function_call: search {"query":"queen gambit"}
iteration #2...
ASSISTANT:
The queen gambit is a chess opening strategy, but based on your course's scope, I should focus on course-related logistics or materials. The lesson emphasizes that off-topic questions (like this one) shouldn't be addressed by the teaching assistant. 

Would you like to explore questions about the course itself instead?



In [79]:
question = "How many lesson pages ?"

result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"number of lesson pages total count"}
iteration #2...
function_call: search {"query":"lesson pages count total"}
iteration #3...
function_call: search {"query": "module structure total lessons count"}
iteration #4...
ASSISTANT:
The course material contains **53 lesson pages** in total.  

Is there anything else you’d like to explore?
